# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/Qwen2.5-0.5B-Instruct`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `qwen-0.5b-brain-v4` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 58 · seq 1024 · linear · 양자화 q4_k_m (데이터 29개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "eyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsoIHsgrDsl5Ag64yA7ZW0IOyVjOugpOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi7KCB7IKwKOqxtOyEpCDqs7XsgqzruYQg7IKw7LacKSDsi5zsiqTthZzsnYQgR0xNIEFQSeyZgCBDbGF1ZGUgQ29kZSDquLDrsJjsnLzroZwg6rCc67CcIOykkS4g66qp7ZGc64qUIOyCrOuejOydtCDshpDsnLzroZwg7ZWY642YIOyggeyCsCDsnpHsl4XsnYQgQUnroZwg7J6Q64+Z7ZmU7ZWY64qUIOqygy4g64yA7Jqp65+JIOuPhOuptC/rrLzrn4kg642w7J207YSwIOyymOumrOulvCDsnITtlbQgQVBJIOyaqeufiSDspp3shKTrj4Qg6rKA7YagIOykkS4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiamVva3NhbiDqtIDroKjtlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJqZW9rc2FuIOyXlOynhOydmCDsoIjrjIAg7JuQ7LmZOiDrqqjrk6Ag7IiY7LmY64qUIO2VqOyImOuhnCDqs4TsgrDtlZjqs6Ag7Iuk7ZaJIOqysOqzvOulvCDsgqzsmqntlZzri6QuIOyerOujjOu5hC/rhbjrrLTruYQv6rK967mEIOu2hO2VtCDqtazsobDrpbwg7KeA7JuQ7ZWY65CYLCDsnoXroKUg64uo6rCA6rCAIOqzteq4ieqwgOyVoSDthrXtlakg64uo6rCA66m0ICfthrXtlakn7Jy866GcIO2RnOq4sO2VmOqzoCDrtoTtlbQg7ZWE7JqU7ISx7J2EIOuztOqzoO2VnOuLpC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiamVva3NhbuydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IlJDIOq4sOuRpSDsoIHsgrAoY29sdW1uLnB5KTog7L2Y7YGs66as7Yq465+JKOOOpSkgPSDri6jrqbTsoIEgw5cg64aS7J20IMOXIOqwnOyImC4g6rGw7ZG47KeRIOuptOyggSjjjqEpID0g65GY66CIIMOXIOuGkuydtCDDlyDqsJzsiJgo67CU64ul66m0IOygnOyZuCkuIOuqqOuToCDsuZjsiJgv6rCc7IiY64qUIOydjOyImCDrtojqsIAsIFZhbHVlRXJyb3LroZwg6rKA7KadLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJqZW9rc2Fu7JeQIOuMgO2VtCDrhKTqsIAg7JWE64qUIOqxuCDrp5DtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IlJDIOuztCDsoIHsgrAoYmVhbS5weSk6IOy9mO2BrOumrO2KuOufiSjjjqUpID0g7Y+tIMOXIOy2pCDDlyDquLjsnbQgw5cg6rCc7IiYLiDqsbDtkbjsp5Eo446hKSA9ICjslpHsuKHrqbQgMsOX7Lakw5fquLjsnbQgKyDrsJHrqbQg7Y+tw5fquLjsnbQpIMOXIOqwnOyImCwg7JyX66m07J2AIOyKrOuemOu4jCDsoJHtlanrtoDrnbwg7KCc7Jm4LiBjb2x1bW4ucHnsnZgg6rO17Ya1IOyXlOynhChJdGVtLCB0b3RhbHMp7J2EIOyerOyCrOyaqS4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiR0JyYWlu7J2EKOulvCkg7KCV66as7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi66Gc7LusIEFJIO2ZmOqyvTog7KeRIOy7tO2TqO2EsOyXkCBHQnJhaW4o6rCc7J24IEFJIOuplOuqqOumrCDsi5zsiqTthZwp7J2EIE9sbGFtYSBnZW1tYTM6MjdiLWl0LXFhdOuhnCDqtazrj5kg7KSR7J206rOgLCBDb25uZWN0IEFJIOyVseycvOuhnCDroZzsu6wgQUkg7YyA7J2EIOq0gOumrC4g7IOB7JeFIOydtOyaqeydtCDsnpDsnKDroZzsmrQgUXdlbiDqs4Tsl7Qg6rK965+JIOuqqOuNuOuhnCDsu6TsiqTthYAgQUnrpbwg7ZWZ7Iq1wrfsp4TtmZQo67OR7ZWpKeyLnO2CrCDqs4Ttmo0uIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6ImdjbS1zeXN0ZW3sl5Ag64yA7ZW0IOyekOyEuO2eiCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6ImdjbS1zeXN0ZW06IOqxtOyEpCDsoIHsgrAo66y865+J7IKw7LacK+qzteyCrOu5hCDqsqzsoIEpIO2UhOuhnOq3uOueqC4gTmVzdEpTK1R5cGVPUk0rUG9zdGdyZVNRTCDrsLHsl5Trk5zroZwgREIg7Jew6rKw6rmM7KeAIOyZhOujjOuQnCDstIjquLAg64uo6rOELCDtlbXsi6wg6riw64qlKOuPhOuptOKGkuusvOufieyCsOy2nOKGkuqyrOyggSnsnYAg66+47LCp7IiYLiDsoITrnrU6ICfsmYTrsr3tlZwg7J6Q64+ZIOusvOufieyCsOy2nCBBSSfrpbwg7LKY7J2M67aA7YSwIOunjOuTpOyngCDslYrqs6Ag64uo6rOE7KCB7Jy866GcIOq1rOy2lSDigJQgMSnsnpDssrQg64+E66m0IO2RnOykgCDthZztlIzrpr8gMilBSSDsnbjsi50g7YyM7J207ZSE65287J24KEdlbWluaS9DbGF1ZGUgQVBJK+uLqOqzhOuzhCDtlITroaztlITtirgpIDMp6rO17KKF67OEIOyCsOy2nCDroZzsp4Eg7JeU7KeEIDQp7JuQ6rCA6rOE7IKwIOyekOuPme2ZlC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiZ2NtLXN5c3RlbeyXkCDrjIDtlbQg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJnY20tc3lzdGVtIOy1nOyihSDsoITrnrUg6rKw7KCVKDIwMjYtMDctMDIpOiAz6rCA7KeAIOyYteyFmCgxLuq4sOyEsSDqtazrj4UgMi7qtazrj4Ur7Luk7Iqk7YSw66eI7J207KeVIDMu7JmE7KCEIOyekOyytOqwnOuwnCkg7KSRICfsmYTsoIQg7J6Q7LK06rCc67CcJyDtmZXsoJUuIOydtOycoDogU21hY29uKO2VmOyasOu5jOuTnCnsnbQg642w7J207YSwIOudveyduOydtCDqsJXtlZjsp4Ag7JWK7JWEIOyInOyImCDruYTsmqkv7Iuc6rCEIOuMgOu5hCDshozsnKDqtowg7Yq466CI7J2065Oc7Jik7ZSE66GcIO2MkOuLqC4g64uk7J2MIOyEuOyFmOydgCDrp4jsiqTthLAg642w7J207YSwIOuqqOuTiOKGkuqyrOyggSDrqqjrk4gg7Iic7Jy866GcIOynhO2WiS4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiU21hY29uIOq0gOugqO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IlNtYWNvbijtlZjsmrDruYzrk5wpIOuypOy5mOuniO2BrCDigJQg7Iuk7KCcIOusvOufieyCsOy2nOyEnCBEQiDsiqTtgqTrp4g6IOy7rOufvOydgCDrj5kv7Li1L+2YuC/si6Qv67aA7JyEL+u2gO2YuC/spJHqs7XsooUv7ZKI66qFL+q3nOqyqS/ri6jsnIQv7IiY65+JLiDrrLjshJwg7Z2Q66aEOiDsiJjrn4nsgrDstpzshJwoaW5zdGFuY2Ug64uo7JyEIOybkOuzuCnihpLrgrTsl63shJwo7J6s66OM67mEL+uFuOustOu5hC/qsr3ruYQg64uo6rCAIOyggeyaqSnihpLsp5Hqs4TtkZwo6riI7JWhIOq4sOykgCDtlansgrAp4oaS7JuQ6rCA6rOE7IKw7IScKOuyleyglSDsmpTsnKgp4oaS6rCR7KeALiDsp5Hqs4TtkZzsnZggJ+yImOufiScg7Lus65+87J2AIO2VreyDgSAxLjAwMCjri6jsnITqsIAg7KCc6rCB6rCB7J206528IOyImOufiSDtlansgrDsnbQg66y07J2Y66+47ZWY6rOgLCDquIjslaEg6riw7KSA7Jy866Gc66eMIFNVTSkuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuybkOqwgOqzhOyCsOydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuybkOqwgOqzhOyCsOyEnCDqsJHsp4Ag67KV7KCVIOqzteyLnSjsl5HshYAg7IiY7Iud7Jy866GcIOqygOymnSDsmYTro4wpOiDqsITsoJHrhbjrrLTruYQ97KeB7KCR64W466y067mEw5c3LjklKOqzoOyglSksIOyCsOyerOuztO2XmOujjD3rhbjrrLTruYTqs4TDlzMuNTYlKOqzoOyglSksIOq1reuvvOyXsOq4iD3sp4HsoJHrhbjrrLTruYTDlzQuNSUsIOq1reuvvOqxtOqwleuztO2XmD3sp4HsoJHrhbjrrLTruYTDlzMuNTQ1JS4g6rOg7Jqp67O07ZeY66OMwrfsgrDsl4XslYjsoITrs7TqsbTqtIDrpqzruYTCt+q4sO2DgOqyveu5hMK37J2867CY6rSA66as67mEwrfsnbTsnKQgNeqwnCDtla3rqqnsnYAg6rO17IKs67mEIOy0neyVoSDqtazqsITrs4Qg64iE7KeE7JqU7JyoLiDqs7XquInqsIDslaE9VFJVTkMo7Iic6rO17IKs67mEK+ydvOuwmOq0gOumrOu5hCvsnbTsnKQsLTUpKDEw66eM7JuQIOuLqOychCDsoIjsgqwpLiDsmpTsnKjtkZzripQg7ZWY65Oc7L2U65Sp7ZWY7KeAIOyViuqzoCDrs4Trj4QgJ+yalOycqCDshKTsoJUg7YWM7J2067iUJ+uhnCDrtoTrpqwg6rSA66asKOunpOuFhCDsoJXrtoAg6rOg7IucIOuzgOqyvSDrjIDsnZEpLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJnY20tc3lzdGVt7JeQIOuMgO2VtCDrhKTqsIAg7JWE64qUIOqxuCDrp5DtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6ImdjbS1zeXN0ZW0g7KGw7KeBL+yCrOyaqeyekCDqtazsobA6IOuplOyduCDsgqzsmqnsnpDripQg6rO166y07YyAKOuzuOu2gCvtmITsnqUg7YyM6rKsIOyWkeyqvSDsobTsnqwpLiDrs7jrtoDripQg7Jes65+sIO2UhOuhnOygne2KuCDrj5nsi5wg7KGw7JyoIOKGkiDrqYDti7DtlITroZzsoJ3tirgg7KeA7JuQ7J20IOyEpOqzhCDsoITsoJwuIO2MgOuzhCjqs7Xsgqwv7JWI7KCEL+2SiOyniC/tmITsnqXsp4Dsm5ApIOuMgOyLnOuztOuTnCDrtoTrpqwgKyDsiqzrnpkv65SU7Iqk7L2U65OcIOyVjOumvCDsl7Drj5kgKyDshJzrpZgt7ZiE7J6lIOu2iOydvOy5mCDsnpDrj5kg7YOQ7KeA6rCAIOuqqe2RnCDsm4ztgaztlIzroZwuIOyjvOugpSDsgqzsl4XsnYAg64yA6riw7JeFIOqzteyepSDtmJHroKXsl4XssrQg65Ox66GdIOyImO2WieydtOudvCDsgrDsl4XshKTruYQg64+E66m0KO2Vjy/tirjroIzsuZgv7Luo67Kg7J207Ja0L+uwsOq0gCkg67mE7KSR7J20IOuGkuydjC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi64+E66m07J247Iud7J2EKOulvCkg7KCV66as7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi64+E66m0IOyekOuPmSDrrLzrn4nsgrDstpwg67Cp7Zal7ISxOiBQREYgVmlzaW9uKO2ZlOyniCDrrLjsoJwpwrdEWEYg7YyM7IuxKOyCrOuejCDqsJzsnoUg7ZWE7JqUKcK3QklNL0lGQyjtmITsnqUg67O06riJ66WgIOuCruydjCkg6rKA7YagIO2bhCwgJ+yekOyytCDrj4TrqbQg7ZGc7KSAIO2FnO2UjOumvyDsoJXrpr3ihpJBSSDsnbjsi50g7YyM7J207ZSE65287J244oaS6rO17KKF67OEIOyCsOy2nCDroZzsp4Eg7JeU7KeEJyDri6jqs4TsoIEg6rWs7LaV7Jy866GcIOy1nOyihSDtlansnZguIOqyve2drOuMgCAyRHRvM0Qg67aE7ISdIOqysOqzvCDrsLHsl5Trk5zqsIAgR2VtaW5pIEFQSSDtmLjstpwg67Cp7IudKOuLqOqzhOuzhCDtlITroaztlITtirg6IE5vcm1hbGl6YXRpb27ihpJTdHJ1Y3R1cmXihpJPcGVuaW5nc+KGklNwYWNlc+KGkkRpbWVuc2lvbnPihpJNYXN0ZXIgSlNPTinsnbTrnbwgZ2NtLXN5c3RlbeuPhCDsu6TsiqTthYAgQ1Yg66qo6424IOyXhuydtCDsnKDsgqwg7YyM7J207ZSE65287J24IOq1rOy2lSDqsIDriqUuIOyYpO2UiOyGjOyKpCDtm4Trs7Q6IEN1YmlDYXNhNUsoNeyynOyepSDrnbzrsqjrp4Eg642w7J207YSw7IWLLCDsgqzsi6Tsg4Eg7ZGc7KSAKSwgVEYyRGVlcEZsb29ycGxhbi4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiZ2NtLXN5c3RlbeyXkCDrjIDtlbQg7J6Q7IS47Z6IIOyVjOugpOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiZ2NtLXN5c3RlbSDquLDsiKAg7Iqk7YOdIOuwjyDsp4Ttlokg7ZiE7ZmpOiDrsLHsl5Trk5wgTmVzdEpTK1R5cGVPUk0rUG9zdGdyZVNRTChEb2NrZXIgNTQzMyDtj6ztirgpLCAuZW52IOyEpOyglSDrsI8gREIg7Jew6rKwIO2ZleyduCDsmYTro4wuIO2UhOuhoO2KuOyXlOuTnCDrr7jssKnsiJguIOuLpOydjCDsnpHsl4U6IOuniOyKpO2EsCDrjbDsnbTthLAg66qo65OIKOqzteyihSDsvZTrk5wv64uo6rCAL+2RnOykgO2SiOyFiCwg6rG07LaVwrfthqDrqqkg7JyE7KO8KeqzvCDqsqzsoIEg66qo65OIKOyeheywsCDtmITsnqUg65Ox66GdL+usvOufiSDsgrDstpwv6rO17IKs67mEIOuCtOyXreyEnCkuIO2UhOuhnOygne2KuCDqsr3roZwgRDovVXNlcnMvdXNlci9EZXNrdG9wL2djbS1zeXN0ZW0uIENsYXVkZSBGYWJsZSA1IOuqqOuNuOydhCAyMDI2LTA3LTA36rmM7KeAIOyCrOyaqSDqsIDriqXtlbQg6re4IOyghOq5jOyngCDshKTqs4Qg7LSI7JWI7J2EIOygleumrO2VmOqzoCAwNy0wN+yXkCDsi6TsoJwg6rWs7ZiEIOyLnOuPhCDqs4Ttmo0uIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuqxsOuyhOuEjOyKpOyXkCDrjIDtlbQg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLruYTsmqkv7IKs7Jqp65+JIOqxsOuyhOuEjOyKpDog67mE6riw7Iig7KeBIOyngeybkOydmCBBUEkg6rO864ukIOyCrOyaqSDsmrDroKQg4oaSIOqysOqzvCDsupDsi7EsIOyCrOyaqeyekC/rtoDshJzrs4Qg7JuU6rCEIO2BrOugiOuUpyDtlZzrj4QsIOq0gOumrOyekCDsgqzsmqnrn4kg64yA7Iuc67O065OcLCDsnqzrtoTshJ0g7ZmV7J24IO2MneyXheycvOuhnCDrjIDsnZEuIOq1reuCtC/tlbTsmbgg7Yis7Yq4656ZIOyghOuetTog6rWt64K0IO2RnOykgCDqs7XsgqzripQg7J6Q7LK0IO2FnO2UjOumvyvtjIzsnbjtipzri53snbQg67mE7JqpIO2aqOycqOyggSwg7ZW07Jm4L+2UjOuenO2KuOuKlCDsmbjqta0gQ0FEIO2RnOykgCDqsJXsoJwg67aI6rCA6528IOuylOyaqSBMTE0gQVBJKEdlbWluaS9DbGF1ZGUpIOuFuOyEoOydtCDsnKDrpqwuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyCrOyXheygnOyViOyEnCDqtIDroKjtlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLqs7zsnbzrho3qsIDshozrk53spp3qsJUg7ZSM656r7Y+8IOyCrOyXheuqqOuNuDog64aN7J6lIOuNsOydtO2EsCvquLDtm4Qg7KCV67O0IOq4sOuwmCDrp57stqQg7J6s67Cw6rCA7J2065OcLCDrho3qsIAt7IaM67mE7J6QIOyngeqxsOuemCDrp4jsvJMsIOuGjeqwgCDsu6TrrqTri4jti7AsIOycoO2GtS/rsLDshqEg7KeA7JuQ7J2EIO2Gte2VqSDsoJzqs7UuIOyImOydteuqqOuNuOydgCDtjJDrp6TsiJjsiJjro4wr7ZSE66as66+47JeEIOq1rOuPheujjCvqtJHqs6Ar66y866WY7KSR6rCc7IiY7IiY66OMLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsgqzsl4XsoJzslYjshJzsnbQo6rCAKSDrrZTsp4Ag7JWM66Ck7KSE656YPyJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJBSSDrj4XqsbDrhbjsnbgg64+M67SEIO2UjOueq+2PvCDsgqzsl4Xrqqjrjbg6IOu5hOyghC/snYzshLEg7J247IudIOq4sOuwmCDruYTsuajsirXsoIEg7Zmc64+ZIOuqqOuLiO2EsOungSwg6rG06rCV642w7J207YSwIOydtOyDgeynle2bhCDqsJDsp4AsIEFJ7LGX67SHIOygleyEnOy8gOyWtCwg7KeA7Jet7IKs7ZqMIOyEnOu5hOyKpCDsl7Dqs4QuIOyImOydteydgCDsp4DsnpDssrQg67O07KGw6riIK+ycoOujjCDrtoDqsIDshJzruYTsiqQrQjJCIOyGlOujqOyFmCDtjJDrp6QuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyCrOyXheygnOyViOyEnOyXkCDrjIDtlbQg64Sk6rCAIOyVhOuKlCDqsbgg66eQ7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJBSSDsp4DsnpDssrQg66+87JuQ7ZW06rKwIO2UjOueq+2PvCDsgqzsl4Xrqqjrjbg6IE5MUCDquLDrsJgg66+87JuQIO2FjeyKpO2KuCDrtoTshJ0g67CPIOyekOuPmeu2hOulmCwgRkFRIOq4sOuwmCDsnpDrj5nsnZHrjIAg7LGX67SHLCDrr7zsm5Ag642w7J207YSwIOyLnOqwge2ZlCDrpqztj6ztirjroZwg7KCV7LGF6rKw7KCVIOyngOybkC4g7IiY7J217J2AIOyngOyekOyytOuzhCDqtazstpUv7Jq07JiBIOqzhOyVveujjCvsgqzsmqnsnpAg65287J207ISg7IqkK+unnuy2pCDsu6jshKTtjIUuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyCrOyXheygnOyViOyEnOydhCjrpbwpIOygleumrO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkFJIOyGjOyDgeqzteyduCDqsr3smIHsp4Dsm5Ag7ZSM656r7Y+8IOyCrOyXheuqqOuNuDog7YyQ66ekL+yImOyalCDsmIjsuKEg6riw67CYIOyerOqzoOq0gOumrCwg6rOg6rCd642w7J207YSwIOu2hOyEnSDqsJzsnbjtmZQg66eI7LyA7YyFLCDsnqzrrLQv7IS466y0IOuNsOydtO2EsCDsl7Drj5ksIOqyveyYgey7qOyEpO2MhSDssZfrtIfsnYQg7Ya17ZWpIOygnOqztS4g7IiY7J217J2AIOyblC/sl7Ag6rWs64+F66OMK+qxsOuemOyImOyImOujjCjshKDtg50pK+q0keqzoCvrjbDsnbTthLDrpqztj6ztirgg7YyQ66ekLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsgqzsl4XsoJzslYjshJzsl5Ag64yA7ZW0IOyekOyEuO2eiCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkFJIOuniOy8gO2MhSDstZzsoIHtmZQg7ZSM656r7Y+8IOyCrOyXheuqqOuNuDog7Jes65+sIOyxhOuEkCjqtJHqs6AvU05TL0NSTSkg642w7J207YSwIO2Gte2Vqeu2hOyEnSwgQUnquLDrsJgg7YOA6rKf6rOg6rCdIOyEuOu2hO2ZlCwg7LGE64SQ67OEIOyEseqzvOyYiOy4oSvsmIjsgrDrsLDrtoQg7LaU7LKcLCDsvZjthZDsuKAg7Zqo6rO87JiI7LihLiBTYWFTIOq1rOuPheujjCvsu6jshKTtjIUr642w7J207YSw66as7Y+s7Yq4IO2MkOunpOuhnCDsiJjsnbXtmZQuIOywqOuzhO2ZlCDtj6zsnbjtirjripQg642w7J207YSwIO2Gte2Vqeq4sOyIoOqzvCDqs6DsoJXrsIAg7JiI7LihIEFJLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsg53shLHtmJVBSeyXkCDrjIDtlbQg7JWM66Ck7KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJBSeqwgCDsp4Hsl4XsnYQg7JmE7KCE7Z6IIOuMgOyytO2VmOq4sOuztOuLpCAn7KeB7JeF7J2YIOuzgO2ZlCfroZwg7J207Ja07KeIIOqwgOuKpeyEseydtCDrhpLsnYwuIOuwmOuzteyggSDri6jsiJzsl4XrrLTripQg7J6Q64+Z7ZmU65CY7KeA66eMIOywveydmOyEscK36rCQ7ISx7KeA64qlwrfrs7XsnqHtlZwg7YyQ64uowrfsnKTrpqzsoIEg6rKw7KCV7J2AIOyXrOyghO2eiCDsnbjqsIQg7JiB7JetLiBBSSDtlITroaztlITtirgg7JeU7KeA64uI7Ja0LCBBSSDsnKTrpqzsoITrrLjqsIAg65OxIOyDiCDsp4HsooXrj4Qg65Ox7J6lLiDsi6TsmqnsoIEg7KCR6re87J2AIEFJ66W8IOuPhOq1rOuhnCDsoIHqt7kg7Zmc7Jqp7ZWY66mwIOyduOqwhOunjOydmCDqsJXsoJDsnYQg6rCc67Cc7ZWY64qUIOqygy4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi7IOd7ISx7ZiVQUkg6rSA66Co7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi7IOd7ISx7ZiVIEFJIOyjvOyalCDsooXrpZg6IO2FjeyKpO2KuCDsg53shLEoQ2hhdEdQVC9HZW1pbmkvQ2xhdWRlIC0g6riA7JOw6riwwrfsmpTslb3Ct+uyiOyXrcK37L2U65SpKSwg7J2066+47KeAIOyDneyEsShEQUxMLUUvTWlkam91cm5leS9TdGFibGUgRGlmZnVzaW9uKSwg7J2M7JWFIOyDneyEsShTdW5vLmFpL0FJVkEvQW1wZXIgTXVzaWMpLCDsmIHsg4Eg7IOd7ISxKFJ1bndheS9TeW50aGVzaWEpLiDstIjrs7TsnpDripQgQ2hhdEdQVCjthY3siqTtirgpwrdEQUxMLUUo7J2066+47KeAKcK3U3Vuby5haSjsnYzslYUp67aA7YSwIOustOujjC/ssrTtl5jtjJDsnLzroZwg7Iuc7J6RIOy2lOyynC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiUGVycGxleGl0eeydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IlBlcnBsZXhpdHkuYWnripQgQUkg6rKA7IOJ7JeU7KeE7Jy866GcLCDri7Xrs4Drp4jri6Qg7Lac7LKYIOunge2BrOulvCDtlajqu5gg7KCc6rO17ZW0IOyLoOuisOyEsSDtmZXsnbjsnbQg6rCA64ql7ZWY6rOgIOyLpOyLnOqwhCDsoJXrs7Qg7KCR6re87J20IOqwleygkC4gRm9jdXMo6rCE6rKw7ZWcIOuLteuzgCkvQ29waWxvdCjsg4HshLgg64yA7ZmU7ZiVKSDrqqjrk5wg7ISg7YOdIOqwgOuKpS4g7LWc7IugIOydtOyKiOuCmCDtlZnsiKAg66as7ISc7LmYIOy0iOq4sCDri6jqs4Qg7KCV67O07IiY7KeR7JeQIO2Kue2eiCDsnKDsmqntlaguIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Iu2UhOuhrO2UhO2KuOyXlOyngOuLiOyWtOungeyXkCDrjIDtlbQg64Sk6rCAIOyVhOuKlCDqsbgg66eQ7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLtmqjqs7zsoIHsnbgg7ZSE66Gs7ZSE7Yq4IOyekeyEsSDsm5DsuZk6ICgxKeq1rOyytOyggeycvOuhnCDsnpHshLEo64yA7IOBwrfrtoTrn4nCt+uCnOydtOuPhCDrqoXsi5wpICgyKeybkO2VmOuKlCDstpzroKUg7ZiV7IudIOyngOyglSjtkZwv66qp66GdIOuTsSkgKDMp7Jet7ZWgIOu2gOyXrCjsmIg6ICfrp4jsvIDtjIUg7KCE66y46rCAIOyeheyepeyXkOyEnCcpICg0KeuLqOqzhCDrqoXsi5woMSkyKTMpIOyInOyEnOuhnCkgKDUp7JiI7IucIOygnOqztS4g6rCZ7J2AIOyjvOygnOuhnCDsl6zrn6wg7ZSE66Gs7ZSE7Yq4IOuwqeyLneydhCDruYTqtZDtlbTrs7TripQg7Jew7Iq17J20IOyLpOugpSDtlqXsg4Hsl5Ag64+E7JuA65CoLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJOb3Rpb24gQUnsnYQo66W8KSDsoJXrpqztlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJOb3Rpb24gQUnripQgL2FpIOuqheugueyWtOuhnCDthY3siqTtirgg7J6s7J6R7ISxLCDsmpTslb0sIOuyiOyXrSwg67iM66CI7J247Iqk7Yag67CNLCDrrLjrspUv6rCA64+F7ISxIOqwnOyEoCDquLDriqXsnYQg66y47IScIOyViOyXkOyEnCDrsJTroZwg7IKs7Jqp7ZWgIOyImCDsnojripQg7IOd7IKw7ISxIOuPhOq1rC4g7YWN7Iqk7Yq4IOyEoO2DnSDtm4Qg7Jqw7YG066at7Jy866Gc64+EIEFJIOuplOuJtCDsoJHqt7wg6rCA64qlLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsg53shLHtmJVBSeyXkCDrjIDtlbQg7J6Q7IS47Z6IIOyVjOugpOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50Ijoi7IOd7ISx7ZiVIEFJ64qUIOyWtOyhsCjthqQp66W8IOyngOygle2VtCDquIDsnYQg7JO4IOyImCDsnojsnYwg4oCUIOyYiDogJ+yghOusuOyggeycvOuhnCcsICfsuZzqt7ztlZjqsownLCAn7Jyg66i465+s7Iqk7ZWY6rKMJyDrk7HsnYQg7ZSE66Gs7ZSE7Yq47JeQIOuqheyLnO2VmOuptCDqsJnsnYAg64K07Jqp64+EIOuLpOuluCDrrLjssrTroZwg7IOd7ISxIOqwgOuKpS4g7YOA6rKfIOuPheyekOyXkCDrp57stpgg7YakIOyhsOygiOydtCDsvZjthZDsuKAg66eI7LyA7YyF7J2YIO2VteyLrCDtmZzsmqnrspUuIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IuyDneyEse2YlUFJ7JeQIOuMgO2VtCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IuyDneyEse2YlSBBSeuhnCDsupDrpq3thLAg7Y6Y66W07IaM64KY66W8IOunjOuTpCDrlYzripQg7J2066aEwrfrgpjsnbTCt+yEseqyqcK366eQ7YiswrfrsLDqsr0g7Iqk7Yag66as66W8IOq1rOyytOyggeycvOuhnCDsp4DsoJXtlbTslbwg7J286rSA65CcIOy6kOumre2EsCDsnKDsp4DqsIAg6rCA64ql7ZWoLiDrp4jsvIDtjIXsmqkg67iM656c65OcIOy6kOumre2EsOuCmCDssZfrtIcg7Y6Y66W07IaM64KYIOyEpOqzhOyXkCDtmZzsmqnrj4TqsIAg64aS7J2MLiJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsg53shLHtmJVBSSDqtIDroKjtlbTshJwg7ISk66qF7ZW07KSYLiJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiLsg53shLHtmJUgQUkg7Zmc7JqpIOy9mO2FkOy4oCDsoJzsnpEg7IKs66GAOiDtmozsnZjroZ0g7J6Q64+Z7KCV66asLCDsnbTrqZTsnbwg7LSI7JWIIOyekeyEsSwg67iU66Gc6re4IOq4gCwg7Jyg7Yqc67iMIOyKpO2BrOumve2KuCwg64W4656YIOyekeyCrC/snpHqs6EsIOydtOuvuOyngCDsg53shLEsIOunjO2ZlCDsi5zrgpjrpqzsmKQsIOyGjOyEpCwg6rSR6rOgIOusuOq1rCwg7IucLCDrj5ntmZQsIOuLpOyWke2VnCDrtoTslbwg64yA67O46rmM7KeAIO2FjeyKpO2KuCDquLDrsJgg7LC97J6RIOyghOuwmOydhCDsu6TrsoTtlaguIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Ik5DU+ydtCjqsIApIOutlOyngCDslYzroKTspITrnpg/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6Ik5DUyjqta3qsIDsp4HrrLTriqXroKXtkZzspIApICfsnLXtlantmJUg7J246rO17KeA64qlIOyghOusuOqwgCcg7KeB66y064qUIDbrjIAg7ZW17IusIOyYgeyXreycvOuhnCDqtazshLE6IEFJ7ZSM656r7Y+8IOq1rOy2lSwgQUnshJzruYTsiqQg6riw7ZqNLCBBSeuqqOuNuOungSwgQUkg7ZWZ7Iq1642w7J207YSwIOq1rOy2lSwgQUnshJzruYTsiqQg7Jq07JiB6rSA66asIOuTsS4g6rCBIOyYgeyXreuniOuLpCDsg53shLHtmJVBSSDsnLXtlanrsKnslYjqs7wg7JaR7J6Q7Lu07ZOo7YyFIOycte2VqeuwqeyViOydhCDrs4Trj4TroZwg7KCc7Iuc7ZWY64qUIOq1rOyhsC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiQUnqtZDsnKHsl5Ag64yA7ZW0IOuEpOqwgCDslYTripQg6rG4IOunkO2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiJ+2AgO2FgCDsgqzsnKAg7ZSE66Gs7ZSE7Yq4JyDsu6TrpqztgZjrn7zsnYAg64yA7KCE7ZmY7Iuc64yA7J2YIEFJIO2AgO2FgOygkO2UhOu2gO2EsCDsi5zsnpHtlbQg7YCA7YWA7KCBIOyCrOqzoCwg7LKg7ZWZ7KCBIO2GteyErSwg7Jqw7KO87KCBIO2GteywsCwgQUkg7YyM7Yq464SI7IutLCBBSS1QQkwg66y47KCc7ZW06rKwLCDsp5Hri6jsp4DtmJwsIO2AgO2FgOyduOulmO2Vmeq5jOyngCDstJ0gMTDrtoDsnpHsnLzroZwg6rWs7ISx65CcIOyyoO2VmcK3QUkg7Jy17ZWpIOq1kOycoSDtlITroZzqt7jrnqguIOyGjO2BrOudvO2FjOyKpCDrrLjri7XrspXsnYQgQUkg7J207ZW07JeQIOyggeyaqe2VmOuKlCDrs4Trj4Qg7Yq4656Z64+EIOyhtOyerC4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi7KeA7Iud7J2EKOulvCkg7KCV66as7ZW07IScIOyEpOuqhe2VtOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiIyDsp4Dsi50ifV19"
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
# qwen Instruct 모델은 내장 chat_template 사용(별도 변환 불필요)
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 58, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "qwen-0.5b-brain-v4"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")
